# Batch Expansion & Azure Classification

This notebook reads all `classified_index` + `stored_results` data from the source DB,
expands every matched snippet to its full surrounding clause **and** classifies it in a
single LLM call (the LLM decides clause boundaries instead of regex), then writes
everything into a new database named `icat-v2-beta`.

Uses DB-backed streaming for checkpoint/resume: a server-side cursor streams source rows,
and a batch existence check against the target DB skips already-processed rows.
No in-memory `processed_ids` set or `work_queue` list.

In [8]:
# Cell 1: Imports & Configuration
import sys
import os
import json
import time
import csv
import gc
import resource
import importlib
from datetime import datetime

# Jupyter sets cwd to the notebook's directory on launch.
# If it's not already data-ingestion/, correct it now.
if os.path.basename(os.getcwd()) != "data-ingestion":
    os.chdir(os.path.join(os.getcwd(), "data-ingestion"))
_nb_dir = os.getcwd()
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()

import insert_data
import second_pipelineoperation
from second_pipelineoperation import (
    expand_and_classify_with_azure,
    expand_and_classify,
    extract_discussion_with_azure,
    extract_metadata_with_indiankanoon,
)
from app import fetch_docmeta

STATS_FILE = "stats_v2_beta.json"
BATCH_SIZE = 10
TARGET_DB_NAME = "icat-v2-beta"
AZURE_INFERENCE_MODEL = os.environ.get("AZURE_INFERENCE_MODEL")

print("Configuration loaded.")
print(f"  Working directory: {os.getcwd()}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  TARGET_DB_NAME = {TARGET_DB_NAME}")
print(f"  STATS_FILE = {STATS_FILE}")
print(f"  MODEL_NAME = {AZURE_INFERENCE_MODEL}")

Configuration loaded.
  Working directory: /home/sankalp-user/Development/ICAT-SLDA/data-ingestion
  BATCH_SIZE = 10
  TARGET_DB_NAME = icat-v2-beta
  STATS_FILE = stats_v2_beta.json
  MODEL_NAME = Llama-3.3-70B-Instruct


In [9]:
# Cell 2: Database Connections

# Source connection (uses env vars as-is)
source_conn = insert_data.create_connection()
assert source_conn is not None, "Failed to connect to source database"
print(f"Connected to source DB: {os.environ.get('DB_NAME')}")

# Create target database if it doesn't exist
admin_uri = (
    f"host={insert_data.DB_HOST} dbname={insert_data.DB_NAME} "
    f"user={insert_data.DB_USER} password={insert_data.DB_PASS} "
    f"sslmode={insert_data.DB_SSLMODE}"
)
admin_conn = psycopg.connect(admin_uri, autocommit=True)
with admin_conn.cursor() as cur:
    try:
        cur.execute(f'CREATE DATABASE "{TARGET_DB_NAME}"')
        print(f"Created database '{TARGET_DB_NAME}'")
    except psycopg.errors.DuplicateDatabase:
        print(f"Database '{TARGET_DB_NAME}' already exists")
admin_conn.close()

# Connect to target database
target_uri = (
    f"host={insert_data.DB_HOST} dbname={TARGET_DB_NAME} "
    f"user={insert_data.DB_USER} password={insert_data.DB_PASS} "
    f"sslmode={insert_data.DB_SSLMODE}"
)
target_conn = psycopg.connect(target_uri)
print(f"Connected to target DB: {TARGET_DB_NAME}")

# Initialize schema in target
insert_data.initialize_db(target_conn)
print("Target DB schema initialized.")

Connected to source DB: icatv1
Database 'icat-v2-beta' already exists
Connected to target DB: icat-v2-beta
Database initialized
Target DB schema initialized.


In [10]:
# Cell 2b: Memory diagnostics

def mem_report(label=""):
    """Print current RSS, virtual size, and peak RSS."""
    try:
        with open('/proc/self/status') as f:
            rss_mb = vm_mb = 0
            for line in f:
                if line.startswith('VmRSS:'):
                    rss_mb = int(line.split()[1]) / 1024
                elif line.startswith('VmSize:'):
                    vm_mb = int(line.split()[1]) / 1024
    except OSError:
        rss_mb = vm_mb = 0
    peak_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    print(f"[MEM {label}] RSS: {rss_mb:.0f} MB  VmSize: {vm_mb:.0f} MB  peak: {peak_mb:.0f} MB")

# Remove any RLIMIT_AS set by previous cell runs (persists across cell re-executions).
# RLIMIT_AS caps virtual address space, NOT physical RAM — the openai/httpx/SSL stack
# uses ~7-8 GB of virtual mappings while only consuming ~150 MB of real memory.
try:
    resource.setrlimit(resource.RLIMIT_AS, (resource.RLIM_INFINITY, resource.RLIM_INFINITY))
    print("RLIMIT_AS cleared (was set by a previous run)")
except (ValueError, resource.error) as e:
    print(f"Could not clear RLIMIT_AS: {e}")

mem_report("baseline after cell 2b")

RLIMIT_AS cleared (was set by a previous run)
[MEM baseline after cell 2b] RSS: 652 MB  VmSize: 4990 MB  peak: 652 MB


In [ ]:
# Cell 3: Test Run — Randomised Sample Across Queries
# Combined pipeline: expand + classify + extract discussion, all via LLM.
# All document text stays in PostgreSQL — only small context windows enter Python.
# Snippet-level results go to a temp table; display is a SQL aggregation.
# No intermediate Python variables for snippets, results, or discussions.

importlib.reload(second_pipelineoperation)

# --- Leak-detection diagnostics ---
_last_rss = [0]  # mutable container so closure can update it

def mem_report(label=""):
    """Print current RSS + peak RSS + delta since last call."""
    try:
        with open('/proc/self/status') as f:
            for line in f:
                if line.startswith('VmRSS:'):
                    current_rss_mb = int(line.split()[1]) / 1024  # kB -> MB
                    break
            else:
                current_rss_mb = 0
    except OSError:
        current_rss_mb = 0
    peak_rss_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    delta = current_rss_mb - _last_rss[0] if _last_rss[0] > 0 else 0
    sign = "+" if delta >= 0 else ""
    print(f"[MEM {label}] current: {current_rss_mb:.0f} MB  peak: {peak_rss_mb:.0f} MB  delta: {sign}{delta:.0f} MB")
    _last_rss[0] = current_rss_mb

# Clear any aborted transaction state on both connections
source_conn.rollback()
target_conn.rollback()

mem_report("cell 3 start")

# Materialize the randomised sample into a temp table on source — one query, no Python lists
with source_conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS _test_sample")
    cur.execute("""
        CREATE TEMP TABLE _test_sample AS
        SELECT doc_id, title, searchquery, matching_indents, matching_columns,
               ROW_NUMBER() OVER () - 1 AS row_idx
        FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY searchquery ORDER BY RANDOM()) AS rn
            FROM classified_index
            WHERE searchquery IN (
                SELECT searchquery FROM (
                    SELECT searchquery
                    FROM classified_index
                    WHERE LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2
                    GROUP BY searchquery
                    ORDER BY RANDOM() LIMIT 5
                ) q
            )
            AND (LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2)
        ) ranked WHERE rn <= 4
    """)
    source_conn.commit()
    cur.execute("SELECT COUNT(*), array_agg(DISTINCT searchquery) FROM _test_sample")
    sample_count, sample_queries = cur.fetchone()

print(f"Randomly selected queries: {sample_queries}")
print(f"Randomly sampled {sample_count} test rows")

# Temp tables for intermediate results (on target_conn so they survive commits)
with target_conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS test_run_snippets")
    cur.execute("DROP TABLE IF EXISTS test_run_rows")
    cur.execute("""
        CREATE TEMP TABLE test_run_snippets (
            row_idx INT, doc_id TEXT, query TEXT, title TEXT,
            snippet_type TEXT, snippet_text TEXT, clause_text TEXT,
            is_contract_clause BOOLEAN, confidence FLOAT, reasoning TEXT
        )
    """)
    cur.execute("""
        CREATE TEMP TABLE test_run_rows (
            row_idx INT, doc_id TEXT, query TEXT, title TEXT,
            discussion TEXT, sentiment TEXT, sentiment_confidence FLOAT,
            is_contract BOOLEAN
        )
    """)
    target_conn.commit()

# Prepare TSV output file
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = f"test_run_{timestamp}.tsv"
FULL_COLUMNS = [
    'Query', 'DocID', 'Title', 'Snippets_MC', 'Snippets_MI',
    'Expanded_MC', 'Expanded_MI', 'Classified_MC', 'Classified_MI',
    'Num_Snippets_MC', 'Num_Snippets_MI', 'Num_Classified_MC', 'Num_Classified_MI',
    'Avg_Classification_Confidence', 'All_Confidences', 'All_Reasonings',
    'Sentiment', 'Sentiment_Confidence', 'Discussion',
    'Is_Contract', 'IndianKanoon_URL',
]
tsv_file = open(csv_path, 'w', newline='')
tsv_writer = csv.DictWriter(tsv_file, fieldnames=FULL_COLUMNS, delimiter='\t')
tsv_writer.writeheader()
tsv_file.flush()

test_errors = []
rows_written = 0

mem_report("before row loop")

# Stream rows from the temp table — no Python list
with source_conn.cursor(name="test_sample_cursor") as src_cur:
    src_cur.itersize = 1
    src_cur.execute("SELECT row_idx, doc_id, title, searchquery, matching_indents, matching_columns FROM _test_sample ORDER BY row_idx")

    for row_idx, doc_id, title, query, mi_raw, mc_raw in src_cur:
        try:
            mc_snippets = insert_data._safe_eval(mc_raw)
            mi_snippets = insert_data._safe_eval(mi_raw)
            print(f"  [Row {row_idx+1}/{sample_count}] doc={doc_id} mc_count={len(mc_snippets)} mi_count={len(mi_snippets)}")
            mem_report(f"row {row_idx+1} start")

            # Process MC snippets — parse and iterate inline, no stored list
            for si, s in enumerate(mc_snippets):
                try:
                    result = expand_and_classify_with_azure(source_conn, doc_id, s)
                except Exception as e:
                    print(f"  [Row {row_idx+1}] expand+classify error for MC snippet in doc {doc_id}: {e}")
                    result = {'clause_text': s, 'is_contract_clause': False,
                              'classification_confidence': 0.0, 'classification_reasoning': f'Error: {e}'}
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO test_run_snippets
                        (row_idx, doc_id, query, title, snippet_type, snippet_text,
                         clause_text, is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,%s,%s,'mc',%s,%s,%s,%s,%s)""",
                        (row_idx, doc_id, query, title, s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                gc.collect()
                # Report every call for first 10, then every 5th
                if si < 10 or (si + 1) % 5 == 0:
                    mem_report(f"row {row_idx+1} after mc[{si+1}/{len(mc_snippets)}]")

            # Process MI snippets — parse and iterate inline
            for si, s in enumerate(mi_snippets):
                try:
                    result = expand_and_classify_with_azure(source_conn, doc_id, s)
                except Exception as e:
                    print(f"  [Row {row_idx+1}] expand+classify error for MI snippet in doc {doc_id}: {e}")
                    result = {'clause_text': s, 'is_contract_clause': False,
                              'classification_confidence': 0.0, 'classification_reasoning': f'Error: {e}'}
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO test_run_snippets
                        (row_idx, doc_id, query, title, snippet_type, snippet_text,
                         clause_text, is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,%s,%s,'mi',%s,%s,%s,%s,%s)""",
                        (row_idx, doc_id, query, title, s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                gc.collect()
                if si < 10 or (si + 1) % 5 == 0:
                    mem_report(f"row {row_idx+1} after mi[{si+1}/{len(mi_snippets)}]")

            # Discussion extraction — fetch first classified clause from DB,
            # call LLM once via walrus operator, write result straight to DB
            with target_conn.cursor() as cur:
                cur.execute("""SELECT clause_text FROM test_run_snippets
                    WHERE row_idx = %s AND is_contract_clause = true LIMIT 1""", (row_idx,))
                clause_row = cur.fetchone()

            if clause_row:
                mem_report(f"row {row_idx+1} before discussion extraction")
                try:
                    with target_conn.cursor() as cur:
                        cur.execute("""INSERT INTO test_run_rows
                            (row_idx, doc_id, query, title, discussion, sentiment,
                             sentiment_confidence, is_contract)
                            VALUES (%s,%s,%s,%s,%s,%s,%s,true)""",
                            (row_idx, doc_id, query, title,
                             (d := extract_discussion_with_azure(source_conn, doc_id, clause_row[0]))['discussion'],
                             d['sentiment'], d['sentiment_confidence']))
                    mem_report(f"row {row_idx+1} after discussion extraction")
                except Exception as e:
                    print(f"  [Row {row_idx+1}] discussion extraction error for doc {doc_id}: {e}")
                    with target_conn.cursor() as cur:
                        cur.execute("""INSERT INTO test_run_rows
                            (row_idx, doc_id, query, title, discussion, sentiment,
                             sentiment_confidence, is_contract)
                            VALUES (%s,%s,%s,%s,'','',0.0,true)""",
                            (row_idx, doc_id, query, title))
            else:
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO test_run_rows
                        (row_idx, doc_id, query, title, discussion, sentiment,
                         sentiment_confidence, is_contract)
                        VALUES (%s,%s,%s,%s,'','',0.0,false)""",
                        (row_idx, doc_id, query, title))
            target_conn.commit()

            # Write TSV row by querying aggregates from temp tables
            with target_conn.cursor() as cur:
                cur.execute("""
                    SELECT
                        array_agg(s.snippet_text) FILTER (WHERE s.snippet_type='mc'),
                        array_agg(s.snippet_text) FILTER (WHERE s.snippet_type='mi'),
                        array_agg(s.clause_text) FILTER (WHERE s.snippet_type='mc'),
                        array_agg(s.clause_text) FILTER (WHERE s.snippet_type='mi'),
                        array_agg(s.clause_text) FILTER (WHERE s.snippet_type='mc' AND s.is_contract_clause),
                        array_agg(s.clause_text) FILTER (WHERE s.snippet_type='mi' AND s.is_contract_clause),
                        COUNT(*) FILTER (WHERE s.snippet_type='mc'),
                        COUNT(*) FILTER (WHERE s.snippet_type='mi'),
                        COUNT(*) FILTER (WHERE s.snippet_type='mc' AND s.is_contract_clause),
                        COUNT(*) FILTER (WHERE s.snippet_type='mi' AND s.is_contract_clause),
                        ROUND(AVG(s.confidence) FILTER (WHERE s.is_contract_clause)::numeric, 4),
                        array_agg(s.confidence),
                        array_agg(s.reasoning),
                        r.sentiment, ROUND(r.sentiment_confidence::numeric, 4),
                        r.discussion, r.is_contract
                    FROM test_run_rows r
                    LEFT JOIN test_run_snippets s USING (row_idx)
                    WHERE r.row_idx = %s
                    GROUP BY r.row_idx, r.sentiment, r.sentiment_confidence,
                             r.discussion, r.is_contract
                """, (row_idx,))
                agg = cur.fetchone()

            tsv_writer.writerow({
                'Query': query, 'DocID': doc_id, 'Title': title or '',
                'Snippets_MC': str(agg[0] or []), 'Snippets_MI': str(agg[1] or []),
                'Expanded_MC': str(agg[2] or []), 'Expanded_MI': str(agg[3] or []),
                'Classified_MC': str(agg[4] or []), 'Classified_MI': str(agg[5] or []),
                'Num_Snippets_MC': agg[6], 'Num_Snippets_MI': agg[7],
                'Num_Classified_MC': agg[8], 'Num_Classified_MI': agg[9],
                'Avg_Classification_Confidence': float(agg[10] or 0),
                'All_Confidences': str(agg[11] or []),
                'All_Reasonings': str(agg[12] or []),
                'Sentiment': agg[13] or '', 'Sentiment_Confidence': float(agg[14] or 0),
                'Discussion': agg[15] or '',
                'Is_Contract': 'Yes' if agg[16] else 'No',
                'IndianKanoon_URL': f'https://indiankanoon.org/doc/{doc_id}/',
            })
            tsv_file.flush()
            rows_written += 1

            print(f"  Row {row_idx+1}/{sample_count}: doc={doc_id} mc={agg[6]} mi={agg[7]} "
                  f"cls_mc={agg[8]} cls_mi={agg[9]} contract={'Yes' if agg[16] else 'No'}")
            mem_report(f"row {row_idx+1} DONE")

        except Exception as e:
            print(f"  [Row {row_idx+1}] SKIPPED doc {doc_id} (query={query}): {e}")
            test_errors.append({'doc_id': doc_id, 'query': query, 'error': str(e)})

        gc.collect()

tsv_file.close()

# Clean up source temp table
with source_conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS _test_sample")
    source_conn.commit()

mem_report("end of cell 3")

# Display summary table via SQL aggregation
df_display = pd.read_sql("""
    SELECT r.query AS "Query", r.doc_id AS "DocID", LEFT(r.title, 50) AS "Title",
           COUNT(*) FILTER (WHERE s.snippet_type='mc') AS "MC",
           COUNT(*) FILTER (WHERE s.snippet_type='mi') AS "MI",
           COUNT(*) FILTER (WHERE s.snippet_type='mc' AND s.is_contract_clause) AS "Cls MC",
           COUNT(*) FILTER (WHERE s.snippet_type='mi' AND s.is_contract_clause) AS "Cls MI",
           ROUND(AVG(s.confidence) FILTER (WHERE s.is_contract_clause)::numeric, 2) AS "Cls Conf",
           r.sentiment AS "Sentiment",
           ROUND(r.sentiment_confidence::numeric, 2) AS "Sent Conf",
           LEFT(r.discussion, 200) AS "Discussion",
           CASE WHEN r.is_contract THEN 'Yes' ELSE 'No' END AS "Contract?"
    FROM test_run_rows r
    LEFT JOIN test_run_snippets s USING (row_idx)
    GROUP BY r.row_idx, r.query, r.doc_id, r.title,
             r.sentiment, r.sentiment_confidence, r.discussion, r.is_contract
    ORDER BY r.row_idx
""", target_conn)
display(df_display)

if test_errors:
    print(f"\n** {len(test_errors)} row(s) failed: **")
    for err in test_errors:
        print(f"  DocID={err['doc_id']}  query={err['query']}  error={err['error']}")

print(f"\nFull details exported to: {csv_path}")
print(f"  {rows_written} rows ({len(test_errors)} errors), {len(FULL_COLUMNS)} columns")
print("\n** Review the table above. Confidence scores should now appear. **")
print("** Do not proceed to full processing until satisfied. **")

In [4]:
# Cell 4: Copy Base Tables to Target

# Clear any aborted transaction state, reload module, ensure tables exist
target_conn.rollback()
importlib.reload(insert_data)
insert_data.initialize_db(target_conn)

# Copy stored_results using a server-side cursor to avoid loading all rows into RAM
inserted_sr = 0
total_sr = 0
with source_conn.cursor(name="copy_stored_results") as src_cur:
    src_cur.itersize = 100
    src_cur.execute("SELECT Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size FROM stored_results")
    batch = []
    for row in src_cur:
        total_sr += 1
        batch.append(row)
        if len(batch) >= 100:
            with target_conn.cursor() as tgt_cur:
                for r in batch:
                    tgt_cur.execute("""
                        INSERT INTO stored_results (Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size)
                        VALUES (%s, %s, %s, %s, %s)
                        ON CONFLICT (Doc_ID) DO NOTHING
                    """, r)
                    inserted_sr += tgt_cur.rowcount
            target_conn.commit()
            batch = []
    # Final partial batch
    if batch:
        with target_conn.cursor() as tgt_cur:
            for r in batch:
                tgt_cur.execute("""
                    INSERT INTO stored_results (Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size)
                    VALUES (%s, %s, %s, %s, %s)
                    ON CONFLICT (Doc_ID) DO NOTHING
                """, r)
                inserted_sr += tgt_cur.rowcount
        target_conn.commit()

print(f"stored_results: {total_sr} source rows, {inserted_sr} newly inserted into target")

# Copy search_queries
with source_conn.cursor() as src_cur:
    src_cur.execute("SELECT searchquery, dateandtime FROM search_queries")
    sq_rows = src_cur.fetchall()

with target_conn.cursor() as tgt_cur:
    for row in sq_rows:
        tgt_cur.execute("""
            INSERT INTO search_queries (searchquery, dateandtime)
            VALUES (%s, %s)
        """, row)
    target_conn.commit()

print(f"search_queries: {len(sq_rows)} rows copied to target")

Database initialized
stored_results: 835 source rows, 0 newly inserted into target
search_queries: 137 rows copied to target


In [6]:
# Cell 5: Extract Court Metadata for stored_results
# Populates court_name, judgment_date, case_citation using Indian Kanoon API.
# Has its own checkpoint file so it can be re-run independently.
# Set FORCE_REPROCESS = True to ignore the checkpoint and re-fetch all docs.

importlib.reload(second_pipelineoperation)
importlib.reload(insert_data)

FORCE_REPROCESS = False   # ← set False to resume from checkpoint
METADATA_CHECKPOINT = "checkpoint_metadata.json"
api_key = os.environ.get("API_KEY")
meta_headers = {'authorization': f"Token {api_key}"}

# Clear any stale transaction state
target_conn.rollback()

# Load metadata checkpoint (ignored if FORCE_REPROCESS)
if not FORCE_REPROCESS and os.path.exists(METADATA_CHECKPOINT):
    with open(METADATA_CHECKPOINT, 'r') as f:
        meta_ckpt = json.load(f)
    meta_processed = set(meta_ckpt.get('processed_doc_ids', []))
    print(f"Metadata checkpoint loaded: {len(meta_processed)} docs already processed")
else:
    meta_processed = set()
    if FORCE_REPROCESS:
        print("FORCE_REPROCESS=True — ignoring checkpoint, re-fetching all docs")
    else:
        print("No metadata checkpoint found. Starting fresh.")

# Find docs needing metadata
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT Doc_ID FROM stored_results
        WHERE court_name IS NULL OR court_name = ''
    """)
    docs_needing_meta = [r[0] for r in cur.fetchall()]

# Filter out already-checkpointed docs
docs_to_process = [d for d in docs_needing_meta if d not in meta_processed]
print(f"{len(docs_to_process)} docs need metadata ({len(docs_needing_meta)} total without, {len(meta_processed)} already done)")

meta_stats = {'api_success': 0, 'failed': 0, 'rows_updated': 0}

for i, doc_id in enumerate(docs_to_process):
    try:
        meta = extract_metadata_with_indiankanoon(doc_id, meta_headers)

        if meta.get('court_name'):
            meta_stats['api_success'] += 1

        # Update stored_results
        with target_conn.cursor() as cur:
            cur.execute("""
                UPDATE stored_results
                SET court_name = %s, judgment_date = %s, case_citation = %s
                WHERE Doc_ID = %s
            """, (meta.get('court_name', ''), meta.get('judgment_date', ''),
                  meta.get('case_citation', ''), doc_id))
            meta_stats['rows_updated'] += cur.rowcount

        meta_processed.add(doc_id)

        # Checkpoint every BATCH_SIZE docs
        if (i + 1) % BATCH_SIZE == 0:
            target_conn.commit()
            with open(METADATA_CHECKPOINT, 'w') as f:
                json.dump({'processed_doc_ids': list(meta_processed),
                           'last_updated': datetime.now().isoformat(),
                           'stats': meta_stats}, f, indent=2)
            print(f"  Metadata: {i + 1}/{len(docs_to_process)} processed "
                  f"(API hits: {meta_stats['api_success']}, rows updated: {meta_stats['rows_updated']})")

        time.sleep(0.2)  # rate limit

    except Exception as e:
        print(f"  Metadata error for {doc_id}: {e}")
        meta_stats['failed'] += 1
        meta_processed.add(doc_id)

# Final commit and checkpoint
target_conn.commit()
with open(METADATA_CHECKPOINT, 'w') as f:
    json.dump({'processed_doc_ids': list(meta_processed),
               'last_updated': datetime.now().isoformat(),
               'stats': meta_stats}, f, indent=2)

print(f"\nMetadata extraction complete:")
print(f"  API success (non-empty court_name): {meta_stats['api_success']}")
print(f"  Rows actually updated in DB:        {meta_stats['rows_updated']}")
print(f"  Failed:                             {meta_stats['failed']}")

Metadata checkpoint loaded: 180 docs already processed
655 docs need metadata (815 total without, 180 already done)
  Metadata: 10/655 processed (API hits: 9, rows updated: 10)
  Metadata: 20/655 processed (API hits: 19, rows updated: 20)
  Metadata: 30/655 processed (API hits: 29, rows updated: 30)
  Metadata: 40/655 processed (API hits: 39, rows updated: 40)
  Metadata: 50/655 processed (API hits: 49, rows updated: 50)
  Metadata: 60/655 processed (API hits: 59, rows updated: 60)
  Metadata: 70/655 processed (API hits: 69, rows updated: 70)
  Metadata: 80/655 processed (API hits: 79, rows updated: 80)
  Metadata: 90/655 processed (API hits: 89, rows updated: 90)
  Metadata: 100/655 processed (API hits: 99, rows updated: 100)
  Metadata: 110/655 processed (API hits: 108, rows updated: 110)
  Metadata: 120/655 processed (API hits: 117, rows updated: 120)
  Metadata: 130/655 processed (API hits: 125, rows updated: 130)
  Metadata: 140/655 processed (API hits: 135, rows updated: 140)
  M

In [30]:
# Cell 7: Streaming Generator — replaces in-memory work_queue
# Set FORCE_REPROCESS = True to re-process all rows, ignoring the target DB checkpoint.

FORCE_REPROCESS = True  # ← set False to resume from checkpoint

def _filter_already_processed(target_conn, rows):
    """Filter out rows whose (doc_id, searchquery) already exist in the target classified_index.

    Uses a VALUES join so the check is a single round-trip per batch.
    Returns only unprocessed rows as dicts with raw (unparsed) snippet strings.
    """
    if not rows:
        return []

    pairs = [(r[0], r[2]) for r in rows]  # (doc_id, searchquery)
    values_placeholder = ", ".join(["(%s, %s)"] * len(pairs))
    query = f"""
        SELECT v.doc_id, v.searchquery
        FROM (VALUES {values_placeholder}) AS v(doc_id, searchquery)
        INNER JOIN classified_index ci
            ON ci.Doc_Id::TEXT = v.doc_id AND ci.searchquery = v.searchquery
    """
    flat_params = [item for pair in pairs for item in pair]

    with target_conn.cursor() as cur:
        cur.execute(query, flat_params)
        existing = set()
        for row in cur:
            existing.add((row[0], row[1]))

    result = []
    for r in rows:
        doc_id, title, searchquery, mi_raw, mc_raw, mc_after_raw, mi_after_raw = r
        if (doc_id, searchquery) not in existing:
            result.append({
                'doc_id': doc_id,
                'title': title,
                'searchquery': searchquery,
                'matching_indents_raw': mi_raw,
                'matching_columns_raw': mc_raw,
                'matching_columns_after_classification_raw': mc_after_raw,
                'matching_indents_after_classification_raw': mi_after_raw,
            })
    return result


def _all_as_dicts(rows):
    """Return all rows as dicts without filtering (used when FORCE_REPROCESS=True)."""
    return [
        {
            'doc_id': r[0],
            'title': r[1],
            'searchquery': r[2],
            'matching_indents_raw': r[3],
            'matching_columns_raw': r[4],
            'matching_columns_after_classification_raw': r[5],
            'matching_indents_after_classification_raw': r[6],
        }
        for r in rows
    ]


def iter_unprocessed_batches(source_conn, target_conn, batch_size, force=False):
    """Generator that streams rows from source classified_index via a server-side cursor
    and yields batches of row dicts. When force=True, skips the already-processed filter.
    """
    with source_conn.cursor(name="stream_classified_index") as src_cur:
        src_cur.itersize = batch_size
        src_cur.execute("""
            SELECT doc_id::TEXT, title, searchquery, matching_indents, matching_columns,
                   matching_columns_after_classification, matching_indents_after_classification
            FROM classified_index
        """)

        batch = []
        for row in src_cur:
            batch.append(row)
            if len(batch) >= batch_size:
                rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
                batch = []
                if rows:
                    yield rows

        # Final partial batch
        if batch:
            rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
            if rows:
                yield rows


# Display source and target counts (no full fetch)
with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_count = cur.fetchone()[0]

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_count = cur.fetchone()[0]

remaining = source_count - target_count
print(f"Source classified_index rows: {source_count}")
print(f"Target classified_index rows: {target_count} (already processed)")
print(f"Approximate remaining: {remaining}")
print(f"FORCE_REPROCESS = {FORCE_REPROCESS}")

Source classified_index rows: 2674
Target classified_index rows: 182 (already processed)
Approximate remaining: 2492
FORCE_REPROCESS = True


In [ ]:
# Cell 8: Process — Expand, Classify & Extract Discussion (main loop)
# Per snippet: Azure expands, local HF classifies (falls back to Azure if conf < 0.65).
# Per row (if contract clauses found): one LLM call for discussion extraction (always Azure).
# Run Cell 6 (stats) and Cell 7 (streaming setup) before this cell.
# Doc_Text never enters Python — all text ops are parameterised SQL inside the functions.
# Snippet results go to a staging temp table; final INSERT is built via SQL aggregation.

importlib.reload(second_pipelineoperation)

def save_stats():
    """Save current stats to stats file."""
    stats['last_updated'] = datetime.now().isoformat()
    with open(STATS_FILE, 'w') as f:
        json.dump(stats, f, indent=2)


def expand_classify_with_retry(conn, doc_id, snippet, max_retries=3):
    """Combined expand+classify with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return expand_and_classify(conn, doc_id, snippet)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


def extract_discussion_with_retry(conn, doc_id, clause, max_retries=3):
    """Extract discussion with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return extract_discussion_with_azure(conn, doc_id, clause)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Discussion retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


# Create staging temp table for per-snippet results (recreated each run)
with target_conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS _staging_snippets")
    cur.execute("""
        CREATE TEMP TABLE _staging_snippets (
            doc_id TEXT, searchquery TEXT,
            snippet_type TEXT, snippet_text TEXT, clause_text TEXT,
            is_contract_clause BOOLEAN, confidence FLOAT, reasoning TEXT
        )
    """)
    target_conn.commit()

print(f"Starting processing (streaming from source, BATCH_SIZE={BATCH_SIZE}, FORCE_REPROCESS={FORCE_REPROCESS})")
batch_num = 0

for batch in iter_unprocessed_batches(source_conn, target_conn, BATCH_SIZE, force=FORCE_REPROCESS):
    batch_num += 1

    for row in batch:
        try:
            # Clear staging for this row
            with target_conn.cursor() as cur:
                cur.execute("DELETE FROM _staging_snippets WHERE doc_id = %s AND searchquery = %s",
                            (row['doc_id'], row['searchquery']))
            target_conn.commit()

            # Process matching_columns snippets — iterate inline, write to staging
            for s in insert_data._safe_eval(row['matching_columns_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mc',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Process matching_indents snippets
            for s in insert_data._safe_eval(row['matching_indents_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mi',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Aggregate from staging via SQL — no Python lists
            with target_conn.cursor() as cur:
                cur.execute("""
                    SELECT
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc'),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi'),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        ROUND(AVG(confidence) FILTER (WHERE is_contract_clause)::numeric, 3),
                        (array_agg(reasoning) FILTER (WHERE is_contract_clause))[1],
                        ROUND(AVG(confidence)::numeric, 3),
                        (array_agg(reasoning))[1],
                        COUNT(*) FILTER (WHERE snippet_type='mc'),
                        COUNT(*) FILTER (WHERE snippet_type='mi'),
                        COUNT(*) FILTER (WHERE is_contract_clause),
                        (array_agg(clause_text) FILTER (WHERE is_contract_clause))[1]
                    FROM _staging_snippets
                    WHERE doc_id = %s AND searchquery = %s
                """, (row['doc_id'], row['searchquery']))
                agg = cur.fetchone()

            # agg indices:
            # 0=expanded_columns, 1=expanded_indents,
            # 2=mc_after, 3=mi_after,
            # 4=expanded_columns_after, 5=expanded_indents_after,
            # 6=avg_classified_conf, 7=first_classified_reasoning,
            # 8=avg_all_conf, 9=first_all_reasoning,
            # 10=num_mc, 11=num_mi, 12=num_classified,
            # 13=first_classified_clause

            avg_confidence = float(agg[6] if agg[6] is not None else agg[8] or 0)
            first_reasoning = agg[7] or agg[9] or ''

            # Extract discussion if any classified clauses exist
            extracted_discussion = ''
            sentiment = ''
            sentiment_conf = 0.0
            if agg[13]:  # first_classified_clause exists
                try:
                    d = extract_discussion_with_retry(source_conn, row['doc_id'], agg[13])
                    extracted_discussion = d['discussion']
                    sentiment = d['sentiment']
                    sentiment_conf = d['sentiment_confidence']
                    stats['total_discussions_extracted'] += 1
                except Exception as e:
                    print(f"  Discussion error for doc_id={row['doc_id']}: {e}")
                time.sleep(0.1)

            # Insert into target classified_index from aggregated staging data
            with target_conn.cursor() as tgt_cur:
                tgt_cur.execute("""
                    INSERT INTO classified_index(
                        Doc_Id, Title, searchquery,
                        matching_indents, matching_columns,
                        matching_columns_after_classification,
                        matching_indents_after_classification,
                        expanded_columns, expanded_indents,
                        expanded_columns_after_classification,
                        expanded_indents_after_classification,
                        extracted_discussion, sentiment,
                        classification_confidence, classification_reasoning,
                        sentiment_confidence
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (Doc_Id, searchquery) DO UPDATE SET
                        Title = EXCLUDED.Title,
                        matching_indents = EXCLUDED.matching_indents,
                        matching_columns = EXCLUDED.matching_columns,
                        matching_columns_after_classification = EXCLUDED.matching_columns_after_classification,
                        matching_indents_after_classification = EXCLUDED.matching_indents_after_classification,
                        expanded_columns = EXCLUDED.expanded_columns,
                        expanded_indents = EXCLUDED.expanded_indents,
                        expanded_columns_after_classification = EXCLUDED.expanded_columns_after_classification,
                        expanded_indents_after_classification = EXCLUDED.expanded_indents_after_classification,
                        extracted_discussion = EXCLUDED.extracted_discussion,
                        sentiment = EXCLUDED.sentiment,
                        classification_confidence = EXCLUDED.classification_confidence,
                        classification_reasoning = EXCLUDED.classification_reasoning,
                        sentiment_confidence = EXCLUDED.sentiment_confidence
                """, (
                    row['doc_id'], row['title'], row['searchquery'],
                    row['matching_indents_raw'], row['matching_columns_raw'],
                    str(agg[2] or []), str(agg[3] or []),
                    str(agg[0] or []), str(agg[1] or []),
                    str(agg[4] or []), str(agg[5] or []),
                    extracted_discussion, sentiment,
                    str(avg_confidence), first_reasoning,
                    str(round(sentiment_conf, 3)),
                ))

            # Track stats
            stats['total_expanded'] += (agg[10] or 0) + (agg[11] or 0)
            stats['total_classified_as_clause'] += agg[12] or 0
            stats['total_processed'] += 1

        except Exception as e:
            print(f"  ERROR processing doc_id={row['doc_id']} query={row['searchquery']}: {e}")
            stats['total_errors'] += 1
            target_conn.rollback()  # clear aborted transaction so next row can proceed

    # Commit and save stats after each batch
    target_conn.commit()
    save_stats()
    print(f"Batch {batch_num} complete "
          f"({stats['total_processed']} total processed, "
          f"{stats['total_discussions_extracted']} discussions, "
          f"{stats['total_errors']} errors)")

print(f"\nProcessing complete. Final stats: {stats}")

Starting processing (streaming from source, BATCH_SIZE=10, FORCE_REPROCESS=True)
[EXTRACT_DISCUSSION] doc_id=9193128, clause_len=1099
[EXTRACT_DISCUSSION] doc_len=48803, raw_val type=<class 'int'>
[EXTRACT_DISCUSSION] scan_len=48803, total_chunks=2, chunk_size=40000
[EXTRACT_DISCUSSION] Processing chunk 1/2: section 1 of 2, start=1, len=40000
[EXTRACT_DISCUSSION]   ✓ Found discussion (confidence=0.90)
[EXTRACT_DISCUSSION] Processing chunk 2/2: section 2 of 2, start=38001, len=10803
[EXTRACT_DISCUSSION]   ✓ Found discussion (confidence=0.90)
[EXTRACT_DISCUSSION] Total discussions found: 2
[EXTRACT_DISCUSSION] Merging 2 discussions
[EXTRACT_DISCUSSION] Merged result: 2 parts
[EXTRACT_DISCUSSION] Returning: sentiment=negative or struck down, confidence=0.9
[EXTRACT_DISCUSSION] doc_id=133653680, clause_len=715
[EXTRACT_DISCUSSION] doc_len=91030, raw_val type=<class 'int'>
[EXTRACT_DISCUSSION] scan_len=91030, total_chunks=3, chunk_size=40000
[EXTRACT_DISCUSSION] Processing chunk 1/3: sectio

In [7]:
# Cell 6: Load Stats

if os.path.exists(STATS_FILE):
    with open(STATS_FILE, 'r') as f:
        stats = json.load(f)
    print(f"Stats loaded: {stats}")
else:
    stats = {
        'total_processed': 0,
        'total_expanded': 0,
        'total_classified_as_clause': 0,
        'total_discussions_extracted': 0,
        'total_errors': 0,
    }
    print("No stats file found. Starting fresh.")

Stats loaded: {'total_processed': 1088, 'total_expanded': 9039, 'total_classified_as_clause': 4851, 'total_discussions_extracted': 920, 'total_errors': 322, 'last_updated': '2026-03-21T09:45:21.187303'}


In [8]:
# Cell 7 - second run: Streaming Generator — replaces in-memory work_queue
# Set FORCE_REPROCESS = True to re-process all rows, ignoring the target DB checkpoint.

FORCE_REPROCESS = False  # ← set False to resume from checkpoint

def _filter_already_processed(target_conn, rows):
    """Filter out rows whose (doc_id, searchquery) already exist in the target classified_index.

    Uses a VALUES join so the check is a single round-trip per batch.
    Returns only unprocessed rows as dicts with raw (unparsed) snippet strings.
    """
    if not rows:
        return []

    pairs = [(r[0], r[2]) for r in rows]  # (doc_id, searchquery)
    values_placeholder = ", ".join(["(%s, %s)"] * len(pairs))
    query = f"""
        SELECT v.doc_id, v.searchquery
        FROM (VALUES {values_placeholder}) AS v(doc_id, searchquery)
        INNER JOIN classified_index ci
            ON ci.Doc_Id::TEXT = v.doc_id AND ci.searchquery = v.searchquery
    """
    flat_params = [item for pair in pairs for item in pair]

    with target_conn.cursor() as cur:
        cur.execute(query, flat_params)
        existing = set()
        for row in cur:
            existing.add((row[0], row[1]))

    result = []
    for r in rows:
        doc_id, title, searchquery, mi_raw, mc_raw, mc_after_raw, mi_after_raw = r
        if (doc_id, searchquery) not in existing:
            result.append({
                'doc_id': doc_id,
                'title': title,
                'searchquery': searchquery,
                'matching_indents_raw': mi_raw,
                'matching_columns_raw': mc_raw,
                'matching_columns_after_classification_raw': mc_after_raw,
                'matching_indents_after_classification_raw': mi_after_raw,
            })
    return result


def _all_as_dicts(rows):
    """Return all rows as dicts without filtering (used when FORCE_REPROCESS=True)."""
    return [
        {
            'doc_id': r[0],
            'title': r[1],
            'searchquery': r[2],
            'matching_indents_raw': r[3],
            'matching_columns_raw': r[4],
            'matching_columns_after_classification_raw': r[5],
            'matching_indents_after_classification_raw': r[6],
        }
        for r in rows
    ]


def iter_unprocessed_batches(source_conn, target_conn, batch_size, force=False):
    """Generator that streams rows from source classified_index via a server-side cursor
    and yields batches of row dicts. When force=True, skips the already-processed filter.
    """
    with source_conn.cursor(name="stream_classified_index") as src_cur:
        src_cur.itersize = batch_size
        src_cur.execute("""
            SELECT doc_id::TEXT, title, searchquery, matching_indents, matching_columns,
                   matching_columns_after_classification, matching_indents_after_classification
            FROM classified_index
        """)

        batch = []
        for row in src_cur:
            batch.append(row)
            if len(batch) >= batch_size:
                rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
                batch = []
                if rows:
                    yield rows

        # Final partial batch
        if batch:
            rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
            if rows:
                yield rows


# Display source and target counts (no full fetch)
with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_count = cur.fetchone()[0]

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_count = cur.fetchone()[0]

remaining = source_count - target_count
print(f"Source classified_index rows: {source_count}")
print(f"Target classified_index rows: {target_count} (already processed)")
print(f"Approximate remaining: {remaining}")
print(f"FORCE_REPROCESS = {FORCE_REPROCESS}")

Source classified_index rows: 2674
Target classified_index rows: 427 (already processed)
Approximate remaining: 2247
FORCE_REPROCESS = False


In [17]:
# Cell 7b - second run: Reset stats from actual DB state
# Corrects the stats file to match what is actually in the target database,
# fixing the discrepancy caused by rollbacks wiping committed rows from stats.
# Note: total_expanded and total_classified_as_clause count individual SNIPPETS
# (not rows), since each row can have multiple matching_columns/matching_indents.
# total_discussions_extracted counts ROWS (one discussion per row).

# Clear any aborted transaction state
target_conn.rollback()

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    db_total_processed = cur.fetchone()[0]

    # Discussions: one per row
    cur.execute("SELECT COUNT(*) FROM classified_index WHERE LENGTH(extracted_discussion) > 0")
    db_total_discussions = cur.fetchone()[0]

    # Expanded: count individual snippets in expanded_columns + expanded_indents
    # Uses LENGTH > 2 to skip empty lists '[]', and splits on delimiter to count items
    DELIM = "', '"
    cur.execute("""SELECT
        COALESCE(SUM(
            CASE WHEN expanded_columns IS NOT NULL AND LENGTH(expanded_columns) > 2
                 THEN array_length(string_to_array(
                     trim(both '[]' from expanded_columns), %s), 1)
                 ELSE 0 END
            +
            CASE WHEN expanded_indents IS NOT NULL AND LENGTH(expanded_indents) > 2
                 THEN array_length(string_to_array(
                     trim(both '[]' from expanded_indents), %s), 1)
                 ELSE 0 END
        ), 0)
        FROM classified_index
    """, (DELIM, DELIM))
    db_total_expanded = cur.fetchone()[0]

    # Classified as clause: count individual snippets in *_after_classification
    cur.execute("""SELECT
        COALESCE(SUM(
            CASE WHEN matching_columns_after_classification IS NOT NULL
                      AND LENGTH(matching_columns_after_classification) > 2
                 THEN array_length(string_to_array(
                     trim(both '[]' from matching_columns_after_classification), %s), 1)
                 ELSE 0 END
            +
            CASE WHEN matching_indents_after_classification IS NOT NULL
                      AND LENGTH(matching_indents_after_classification) > 2
                 THEN array_length(string_to_array(
                     trim(both '[]' from matching_indents_after_classification), %s), 1)
                 ELSE 0 END
        ), 0)
        FROM classified_index
    """, (DELIM, DELIM))
    db_total_classified = cur.fetchone()[0]

    cur.execute("SELECT COUNT(*) FROM classified_index WHERE classification_reasoning LIKE '%%Error%%'")
    db_total_errors = cur.fetchone()[0]

print("Stats file vs actual DB:")
field_hdr = "Field"
stats_hdr = "Stats file"
db_hdr = "Actual DB"
diff_hdr = "Diff"
print(f"  {field_hdr:<40} {stats_hdr:>12} {db_hdr:>12} {diff_hdr:>8}")
print("-" * 75)
for key, db_val, note in [
    ("total_processed", db_total_processed, "rows"),
    ("total_expanded", db_total_expanded, "snippets"),
    ("total_classified_as_clause", db_total_classified, "snippets"),
    ("total_discussions_extracted", db_total_discussions, "rows"),
    ("total_errors", db_total_errors, "rows"),
]:
    old_val = stats.get(key, 0)
    diff = db_val - old_val
    label = f"{key} ({note})"
    print(f"  {label:<40} {old_val:>12} {db_val:>12} {diff:>+8}")

# Reset stats to match DB
stats["total_processed"] = db_total_processed
stats["total_expanded"] = db_total_expanded
stats["total_classified_as_clause"] = db_total_classified
stats["total_discussions_extracted"] = db_total_discussions
stats["total_errors"] = db_total_errors
stats["last_updated"] = datetime.now().isoformat()

with open(STATS_FILE, "w") as f:
    json.dump(stats, f, indent=2)

print(f"\nStats file reset to match actual DB state.")
print(f"Stats: {stats}")


Stats file vs actual DB:
  Field                                      Stats file    Actual DB     Diff
---------------------------------------------------------------------------
  total_processed (rows)                            427          427       +0
  total_expanded (snippets)                         518         3504    +2986
  total_classified_as_clause (snippets)             368         2028    +1660
  total_discussions_extracted (rows)                359          359       +0
  total_errors (rows)                                 0            0       +0

Stats file reset to match actual DB state.
Stats: {'total_processed': 427, 'total_expanded': 3504, 'total_classified_as_clause': 2028, 'total_discussions_extracted': 359, 'total_errors': 0, 'last_updated': '2026-03-21T10:38:30.263372'}


In [ ]:
# Cell 8-second run: Process — Expand, Classify & Extract Discussion (main loop)
# Per snippet: Azure expands, local HF classifies (falls back to Azure if conf < 0.65).
# Per row (if contract clauses found): one LLM call for discussion extraction (always Azure).
# Run Cell 6 (stats) and Cell 7 (streaming setup) before this cell.
# Doc_Text never enters Python — all text ops are parameterised SQL inside the functions.
# Snippet results go to a staging temp table; final INSERT is built via SQL aggregation.

importlib.reload(second_pipelineoperation)

def save_stats():
    """Save current stats to stats file."""
    stats['last_updated'] = datetime.now().isoformat()
    with open(STATS_FILE, 'w') as f:
        json.dump(stats, f, indent=2)


def expand_classify_with_retry(conn, doc_id, snippet, max_retries=3):
    """Combined expand+classify with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return expand_and_classify(conn, doc_id, snippet)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


def extract_discussion_with_retry(conn, doc_id, clause, max_retries=3):
    """Extract discussion with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return extract_discussion_with_azure(conn, doc_id, clause)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Discussion retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


# Create staging temp table for per-snippet results (recreated each run)
with target_conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS _staging_snippets")
    cur.execute("""
        CREATE TEMP TABLE _staging_snippets (
            doc_id TEXT, searchquery TEXT,
            snippet_type TEXT, snippet_text TEXT, clause_text TEXT,
            is_contract_clause BOOLEAN, confidence FLOAT, reasoning TEXT
        )
    """)
    target_conn.commit()

print(f"Starting processing (streaming from source, BATCH_SIZE={BATCH_SIZE}, FORCE_REPROCESS={FORCE_REPROCESS})")
batch_num = 0

for batch in iter_unprocessed_batches(source_conn, target_conn, BATCH_SIZE, force=FORCE_REPROCESS):
    batch_num += 1

    for row in batch:
        try:
            # Clear staging for this row
            with target_conn.cursor() as cur:
                cur.execute("DELETE FROM _staging_snippets WHERE doc_id = %s AND searchquery = %s",
                            (row['doc_id'], row['searchquery']))
            target_conn.commit()

            # Process matching_columns snippets — iterate inline, write to staging
            for s in insert_data._safe_eval(row['matching_columns_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mc',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Process matching_indents snippets
            for s in insert_data._safe_eval(row['matching_indents_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mi',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Aggregate from staging via SQL — no Python lists
            with target_conn.cursor() as cur:
                cur.execute("""
                    SELECT
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc'),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi'),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        ROUND(AVG(confidence) FILTER (WHERE is_contract_clause)::numeric, 3),
                        (array_agg(reasoning) FILTER (WHERE is_contract_clause))[1],
                        ROUND(AVG(confidence)::numeric, 3),
                        (array_agg(reasoning))[1],
                        COUNT(*) FILTER (WHERE snippet_type='mc'),
                        COUNT(*) FILTER (WHERE snippet_type='mi'),
                        COUNT(*) FILTER (WHERE is_contract_clause),
                        (array_agg(clause_text) FILTER (WHERE is_contract_clause))[1]
                    FROM _staging_snippets
                    WHERE doc_id = %s AND searchquery = %s
                """, (row['doc_id'], row['searchquery']))
                agg = cur.fetchone()

            # agg indices:
            # 0=expanded_columns, 1=expanded_indents,
            # 2=mc_after, 3=mi_after,
            # 4=expanded_columns_after, 5=expanded_indents_after,
            # 6=avg_classified_conf, 7=first_classified_reasoning,
            # 8=avg_all_conf, 9=first_all_reasoning,
            # 10=num_mc, 11=num_mi, 12=num_classified,
            # 13=first_classified_clause

            avg_confidence = float(agg[6] if agg[6] is not None else agg[8] or 0)
            first_reasoning = agg[7] or agg[9] or ''

            # Extract discussion if any classified clauses exist
            extracted_discussion = ''
            sentiment = ''
            sentiment_conf = 0.0
            if agg[13]:  # first_classified_clause exists
                try:
                    d = extract_discussion_with_retry(source_conn, row['doc_id'], agg[13])
                    extracted_discussion = d['discussion']
                    sentiment = d['sentiment']
                    sentiment_conf = d['sentiment_confidence']
                    stats['total_discussions_extracted'] += 1
                except Exception as e:
                    print(f"  Discussion error for doc_id={row['doc_id']}: {e}")
                time.sleep(0.1)

            # Insert into target classified_index from aggregated staging data
            with target_conn.cursor() as tgt_cur:
                tgt_cur.execute("""
                    INSERT INTO classified_index(
                        Doc_Id, Title, searchquery,
                        matching_indents, matching_columns,
                        matching_columns_after_classification,
                        matching_indents_after_classification,
                        expanded_columns, expanded_indents,
                        expanded_columns_after_classification,
                        expanded_indents_after_classification,
                        extracted_discussion, sentiment,
                        classification_confidence, classification_reasoning,
                        sentiment_confidence
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (Doc_Id, searchquery) DO UPDATE SET
                        Title = EXCLUDED.Title,
                        matching_indents = EXCLUDED.matching_indents,
                        matching_columns = EXCLUDED.matching_columns,
                        matching_columns_after_classification = EXCLUDED.matching_columns_after_classification,
                        matching_indents_after_classification = EXCLUDED.matching_indents_after_classification,
                        expanded_columns = EXCLUDED.expanded_columns,
                        expanded_indents = EXCLUDED.expanded_indents,
                        expanded_columns_after_classification = EXCLUDED.expanded_columns_after_classification,
                        expanded_indents_after_classification = EXCLUDED.expanded_indents_after_classification,
                        extracted_discussion = EXCLUDED.extracted_discussion,
                        sentiment = EXCLUDED.sentiment,
                        classification_confidence = EXCLUDED.classification_confidence,
                        classification_reasoning = EXCLUDED.classification_reasoning,
                        sentiment_confidence = EXCLUDED.sentiment_confidence
                """, (
                    row['doc_id'], row['title'], row['searchquery'],
                    row['matching_indents_raw'], row['matching_columns_raw'],
                    str(agg[2] or []), str(agg[3] or []),
                    str(agg[0] or []), str(agg[1] or []),
                    str(agg[4] or []), str(agg[5] or []),
                    extracted_discussion, sentiment,
                    str(avg_confidence), first_reasoning,
                    str(round(sentiment_conf, 3)),
                ))

            # Track stats
            stats['total_expanded'] += (agg[10] or 0) + (agg[11] or 0)
            stats['total_classified_as_clause'] += agg[12] or 0
            stats['total_processed'] += 1

        except Exception as e:
            print(f"  ERROR processing doc_id={row['doc_id']} query={row['searchquery']}: {e}")
            stats['total_errors'] += 1
            target_conn.rollback()  # clear aborted transaction so next row can proceed

    # Commit and save stats after each batch
    target_conn.commit()
    save_stats()
    print(f"Batch {batch_num} complete "
          f"({stats['total_processed']} total processed, "
          f"{stats['total_discussions_extracted']} discussions, "
          f"{stats['total_errors']} errors)")

print(f"\nProcessing complete. Final stats: {stats}")

Starting processing (streaming from source, BATCH_SIZE=10, FORCE_REPROCESS=False)
  Retry 1/3 after 1s: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
  Retry 2/3 after 2s: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify

In [5]:
# Cell 6/7/7b — third run: Load stats from DB + streaming setup
# Combined cell that loads stats directly from the database (not the checkpoint
# file), sets up the streaming generator, and displays remaining work.

import importlib
importlib.reload(second_pipelineoperation)
from second_pipelineoperation import expand_and_classify, extract_discussion_with_azure

# Clear any aborted transaction state
source_conn.rollback()
target_conn.rollback()

# ── Load stats from actual DB state ──────────────────────────────────────────

DELIM = "', '"

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    db_processed = cur.fetchone()[0]

    cur.execute("SELECT COUNT(*) FROM classified_index WHERE LENGTH(extracted_discussion) > 0")
    db_discussions = cur.fetchone()[0]

    cur.execute("""SELECT COALESCE(SUM(
        CASE WHEN expanded_columns IS NOT NULL AND LENGTH(expanded_columns) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from expanded_columns), %s), 1)
             ELSE 0 END
        +
        CASE WHEN expanded_indents IS NOT NULL AND LENGTH(expanded_indents) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from expanded_indents), %s), 1)
             ELSE 0 END
    ), 0) FROM classified_index""", (DELIM, DELIM))
    db_expanded = cur.fetchone()[0]

    cur.execute("""SELECT COALESCE(SUM(
        CASE WHEN matching_columns_after_classification IS NOT NULL
                  AND LENGTH(matching_columns_after_classification) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from matching_columns_after_classification), %s), 1)
             ELSE 0 END
        +
        CASE WHEN matching_indents_after_classification IS NOT NULL
                  AND LENGTH(matching_indents_after_classification) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from matching_indents_after_classification), %s), 1)
             ELSE 0 END
    ), 0) FROM classified_index""", (DELIM, DELIM))
    db_classified = cur.fetchone()[0]

    cur.execute("SELECT COUNT(*) FROM classified_index WHERE classification_reasoning LIKE '%%Error%%'")
    db_errors = cur.fetchone()[0]

stats = {
    'total_processed': db_processed,
    'total_expanded': db_expanded,
    'total_classified_as_clause': db_classified,
    'total_discussions_extracted': db_discussions,
    'total_errors': db_errors,
    'last_updated': datetime.now().isoformat(),
}

with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2)

print("Stats loaded from DB:")
print(f"  total_processed (rows):              {db_processed}")
print(f"  total_expanded (snippets):           {db_expanded}")
print(f"  total_classified_as_clause (snippets): {db_classified}")
print(f"  total_discussions_extracted (rows):   {db_discussions}")
print(f"  total_errors (rows):                 {db_errors}")

# ── Streaming generator ──────────────────────────────────────────────────────

FORCE_REPROCESS = False

def _filter_already_processed(target_conn, rows):
    """Filter out rows whose (doc_id, searchquery) already exist in target."""
    if not rows:
        return []
    pairs = [(r[0], r[2]) for r in rows]
    values_placeholder = ", ".join(["(%s, %s)"] * len(pairs))
    query = f"""
        SELECT v.doc_id, v.searchquery
        FROM (VALUES {values_placeholder}) AS v(doc_id, searchquery)
        INNER JOIN classified_index ci
            ON ci.Doc_Id::TEXT = v.doc_id AND ci.searchquery = v.searchquery
    """
    flat_params = [item for pair in pairs for item in pair]
    with target_conn.cursor() as cur:
        cur.execute(query, flat_params)
        existing = set(cur.fetchall())
    return [
        {
            'doc_id': r[0], 'title': r[1], 'searchquery': r[2],
            'matching_indents_raw': r[3], 'matching_columns_raw': r[4],
            'matching_columns_after_classification_raw': r[5],
            'matching_indents_after_classification_raw': r[6],
        }
        for r in rows if (r[0], r[2]) not in existing
    ]


def _all_as_dicts(rows):
    """Return all rows as dicts without filtering (FORCE_REPROCESS mode)."""
    return [
        {
            'doc_id': r[0], 'title': r[1], 'searchquery': r[2],
            'matching_indents_raw': r[3], 'matching_columns_raw': r[4],
            'matching_columns_after_classification_raw': r[5],
            'matching_indents_after_classification_raw': r[6],
        }
        for r in rows
    ]


def iter_unprocessed_batches(source_conn, target_conn, batch_size, force=False):
    """Stream rows from source, yield batches of unprocessed row dicts."""
    with source_conn.cursor(name="stream_classified_index") as src_cur:
        src_cur.itersize = batch_size
        src_cur.execute("""
            SELECT doc_id::TEXT, title, searchquery, matching_indents, matching_columns,
                   matching_columns_after_classification, matching_indents_after_classification
            FROM classified_index
        """)
        batch = []
        for row in src_cur:
            batch.append(row)
            if len(batch) >= batch_size:
                rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
                batch = []
                if rows:
                    yield rows
        if batch:
            rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
            if rows:
                yield rows


# ── Display remaining work ───────────────────────────────────────────────────

with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_count = cur.fetchone()[0]

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_count = cur.fetchone()[0]

remaining = source_count - target_count
print(f"\nSource classified_index rows: {source_count}")
print(f"Target classified_index rows: {target_count} (already processed)")
print(f"Remaining: {remaining}")
print(f"FORCE_REPROCESS = {FORCE_REPROCESS}")

Stats loaded from DB:
  total_processed (rows):              507
  total_expanded (snippets):           4109
  total_classified_as_clause (snippets): 2444
  total_discussions_extracted (rows):   426
  total_errors (rows):                 0

Source classified_index rows: 2674
Target classified_index rows: 507 (already processed)
Remaining: 2167
FORCE_REPROCESS = False


In [6]:
# Cell 8 — third run: Process — Expand, Classify & Extract Discussion
# FIX: commits after EACH row's classified_index INSERT (not per-batch),
# preventing rollbacks from wiping successfully processed rows.
# Uses expand_and_classify and extract_discussion_with_azure
# from second_pipelineoperation (reloaded in Cell 13).

def save_stats():
    """Save current stats to stats file."""
    stats['last_updated'] = datetime.now().isoformat()
    with open(STATS_FILE, 'w') as f:
        json.dump(stats, f, indent=2)


def expand_classify_with_retry(conn, doc_id, snippet, max_retries=3):
    """Combined expand+classify with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return expand_and_classify(conn, doc_id, snippet)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


def extract_discussion_with_retry(conn, doc_id, clause, max_retries=3):
    """Extract discussion with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return extract_discussion_with_azure(conn, doc_id, clause)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Discussion retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


# Create staging temp table for per-snippet results (recreated each run)
with target_conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS _staging_snippets")
    cur.execute("""
        CREATE TEMP TABLE _staging_snippets (
            doc_id TEXT, searchquery TEXT,
            snippet_type TEXT, snippet_text TEXT, clause_text TEXT,
            is_contract_clause BOOLEAN, confidence FLOAT, reasoning TEXT
        )
    """)
    target_conn.commit()

print(f"Starting processing (streaming from source, BATCH_SIZE={BATCH_SIZE}, FORCE_REPROCESS={FORCE_REPROCESS})")
batch_num = 0

for batch in iter_unprocessed_batches(source_conn, target_conn, BATCH_SIZE, force=FORCE_REPROCESS):
    batch_num += 1

    for row in batch:
        try:
            # Clear staging for this row
            with target_conn.cursor() as cur:
                cur.execute("DELETE FROM _staging_snippets WHERE doc_id = %s AND searchquery = %s",
                            (row['doc_id'], row['searchquery']))
            target_conn.commit()

            # Process matching_columns snippets — iterate inline, write to staging
            for s in insert_data._safe_eval(row['matching_columns_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mc',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Process matching_indents snippets
            for s in insert_data._safe_eval(row['matching_indents_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mi',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Aggregate from staging via SQL — no Python lists
            with target_conn.cursor() as cur:
                cur.execute("""
                    SELECT
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc'),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi'),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        ROUND(AVG(confidence) FILTER (WHERE is_contract_clause)::numeric, 3),
                        (array_agg(reasoning) FILTER (WHERE is_contract_clause))[1],
                        ROUND(AVG(confidence)::numeric, 3),
                        (array_agg(reasoning))[1],
                        COUNT(*) FILTER (WHERE snippet_type='mc'),
                        COUNT(*) FILTER (WHERE snippet_type='mi'),
                        COUNT(*) FILTER (WHERE is_contract_clause),
                        (array_agg(clause_text) FILTER (WHERE is_contract_clause))[1]
                    FROM _staging_snippets
                    WHERE doc_id = %s AND searchquery = %s
                """, (row['doc_id'], row['searchquery']))
                agg = cur.fetchone()

            avg_confidence = float(agg[6] if agg[6] is not None else agg[8] or 0)
            first_reasoning = agg[7] or agg[9] or ''

            # Extract discussion if any classified clauses exist
            extracted_discussion = ''
            sentiment = ''
            sentiment_conf = 0.0
            if agg[13]:  # first_classified_clause exists
                try:
                    d = extract_discussion_with_retry(source_conn, row['doc_id'], agg[13])
                    extracted_discussion = d['discussion']
                    sentiment = d['sentiment']
                    sentiment_conf = d['sentiment_confidence']
                    stats['total_discussions_extracted'] += 1
                except Exception as e:
                    print(f"  Discussion error for doc_id={row['doc_id']}: {e}")
                time.sleep(0.1)

            # Insert into target classified_index
            with target_conn.cursor() as tgt_cur:
                tgt_cur.execute("""
                    INSERT INTO classified_index(
                        Doc_Id, Title, searchquery,
                        matching_indents, matching_columns,
                        matching_columns_after_classification,
                        matching_indents_after_classification,
                        expanded_columns, expanded_indents,
                        expanded_columns_after_classification,
                        expanded_indents_after_classification,
                        extracted_discussion, sentiment,
                        classification_confidence, classification_reasoning,
                        sentiment_confidence
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (Doc_Id, searchquery) DO UPDATE SET
                        Title = EXCLUDED.Title,
                        matching_indents = EXCLUDED.matching_indents,
                        matching_columns = EXCLUDED.matching_columns,
                        matching_columns_after_classification = EXCLUDED.matching_columns_after_classification,
                        matching_indents_after_classification = EXCLUDED.matching_indents_after_classification,
                        expanded_columns = EXCLUDED.expanded_columns,
                        expanded_indents = EXCLUDED.expanded_indents,
                        expanded_columns_after_classification = EXCLUDED.expanded_columns_after_classification,
                        expanded_indents_after_classification = EXCLUDED.expanded_indents_after_classification,
                        extracted_discussion = EXCLUDED.extracted_discussion,
                        sentiment = EXCLUDED.sentiment,
                        classification_confidence = EXCLUDED.classification_confidence,
                        classification_reasoning = EXCLUDED.classification_reasoning,
                        sentiment_confidence = EXCLUDED.sentiment_confidence
                """, (
                    row['doc_id'], row['title'], row['searchquery'],
                    row['matching_indents_raw'], row['matching_columns_raw'],
                    str(agg[2] or []), str(agg[3] or []),
                    str(agg[0] or []), str(agg[1] or []),
                    str(agg[4] or []), str(agg[5] or []),
                    extracted_discussion, sentiment,
                    str(avg_confidence), first_reasoning,
                    str(round(sentiment_conf, 3)),
                ))

            # FIX: commit immediately after each row's INSERT
            # Prevents a later row's error from rolling back this row
            target_conn.commit()

            # Track stats (after commit, so stats only count persisted rows)
            stats['total_expanded'] += (agg[10] or 0) + (agg[11] or 0)
            stats['total_classified_as_clause'] += agg[12] or 0
            stats['total_processed'] += 1

        except Exception as e:
            print(f"  ERROR processing doc_id={row['doc_id']} query={row['searchquery']}: {e}")
            stats['total_errors'] += 1
            target_conn.rollback()  # clear aborted transaction so next row can proceed

    # Save stats after each batch (commit already happened per-row above)
    save_stats()
    print(f"Batch {batch_num} complete "
          f"({stats['total_processed']} total processed, "
          f"{stats['total_discussions_extracted']} discussions, "
          f"{stats['total_errors']} errors)")

print(f"\nProcessing complete. Final stats: {stats}")


Starting processing (streaming from source, BATCH_SIZE=10, FORCE_REPROCESS=False)
  Retry 1/3 after 1s: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
  Retry 2/3 after 2s: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify

KeyboardInterrupt: 

In [13]:
# Cell 6/7/7b — fourth run: Load stats from DB + streaming setup
# Combined cell that loads stats directly from the database (not the checkpoint
# file), sets up the streaming generator, and displays remaining work.
# Fourth run: Similar to third run, for continued processing of remaining rows.

import importlib
importlib.reload(second_pipelineoperation)
from second_pipelineoperation import expand_and_classify, extract_discussion_with_azure

# Clear any aborted transaction state
source_conn.rollback()
target_conn.rollback()

# ── Load stats from actual DB state ──────────────────────────────────────────

DELIM = "', '"

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    db_processed = cur.fetchone()[0]

    cur.execute("SELECT COUNT(*) FROM classified_index WHERE LENGTH(extracted_discussion) > 0")
    db_discussions = cur.fetchone()[0]

    cur.execute("""SELECT COALESCE(SUM(
        CASE WHEN expanded_columns IS NOT NULL AND LENGTH(expanded_columns) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from expanded_columns), %s), 1)
             ELSE 0 END
        +
        CASE WHEN expanded_indents IS NOT NULL AND LENGTH(expanded_indents) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from expanded_indents), %s), 1)
             ELSE 0 END
    ), 0) FROM classified_index""", (DELIM, DELIM))
    db_expanded = cur.fetchone()[0]

    cur.execute("""SELECT COALESCE(SUM(
        CASE WHEN matching_columns_after_classification IS NOT NULL
                  AND LENGTH(matching_columns_after_classification) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from matching_columns_after_classification), %s), 1)
             ELSE 0 END
        +
        CASE WHEN matching_indents_after_classification IS NOT NULL
                  AND LENGTH(matching_indents_after_classification) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from matching_indents_after_classification), %s), 1)
             ELSE 0 END
    ), 0) FROM classified_index""", (DELIM, DELIM))
    db_classified = cur.fetchone()[0]

    cur.execute("SELECT COUNT(*) FROM classified_index WHERE classification_reasoning LIKE '%%Error%%'")
    db_errors = cur.fetchone()[0]

stats = {
    'total_processed': db_processed,
    'total_expanded': db_expanded,
    'total_classified_as_clause': db_classified,
    'total_discussions_extracted': db_discussions,
    'total_errors': db_errors,
    'last_updated': datetime.now().isoformat(),
}

with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2)

print("Stats loaded from DB (fourth run):")
print(f"  total_processed (rows):              {db_processed}")
print(f"  total_expanded (snippets):           {db_expanded}")
print(f"  total_classified_as_clause (snippets): {db_classified}")
print(f"  total_discussions_extracted (rows):   {db_discussions}")
print(f"  total_errors (rows):                 {db_errors}")

# ── Streaming generator ──────────────────────────────────────────────────────

FORCE_REPROCESS = False

def _filter_already_processed(target_conn, rows):
    """Filter out rows whose (doc_id, searchquery) already exist in target."""
    if not rows:
        return []
    pairs = [(r[0], r[2]) for r in rows]
    values_placeholder = ", ".join(["(%s, %s)"] * len(pairs))

    query = f"""
        SELECT v.doc_id, v.searchquery
        FROM (VALUES {values_placeholder}) AS v(doc_id, searchquery)
        INNER JOIN classified_index ci
            ON ci.Doc_Id::TEXT = v.doc_id AND ci.searchquery = v.searchquery
    """
    flat_params = [item for pair in pairs for item in pair]
    with target_conn.cursor() as cur:
        cur.execute(query, flat_params)
        existing = set(cur.fetchall())
    return [
        {
            'doc_id': r[0], 'title': r[1], 'searchquery': r[2],
            'matching_indents_raw': r[3], 'matching_columns_raw': r[4],
            'matching_columns_after_classification_raw': r[5],
            'matching_indents_after_classification_raw': r[6],
        }
        for r in rows if (r[0], r[2]) not in existing
    ]


def _all_as_dicts(rows):
    """Return all rows as dicts without filtering (FORCE_REPROCESS mode)."""
    return [
        {
            'doc_id': r[0], 'title': r[1], 'searchquery': r[2],
            'matching_indents_raw': r[3], 'matching_columns_raw': r[4],
            'matching_columns_after_classification_raw': r[5],
            'matching_indents_after_classification_raw': r[6],
        }
        for r in rows
    ]


def iter_unprocessed_batches(source_conn, target_conn, batch_size, force=False):
    """Stream rows from source, yield batches of unprocessed row dicts."""
    with source_conn.cursor(name="stream_classified_index_fourth") as src_cur:
        src_cur.itersize = batch_size
        src_cur.execute("""
            SELECT doc_id::TEXT, title, searchquery, matching_indents, matching_columns,
                   matching_columns_after_classification, matching_indents_after_classification
            FROM classified_index
        """)
        batch = []
        for row in src_cur:
            batch.append(row)
            if len(batch) >= batch_size:
                rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
                batch = []
                if rows:
                    yield rows
        if batch:
            rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
            if rows:
                yield rows


# ── Display remaining work ───────────────────────────────────────────────────

with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_count = cur.fetchone()[0]

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_count = cur.fetchone()[0]

remaining = source_count - target_count
print(f"\nFourth run status:")
print(f"  Source classified_index rows: {source_count}")
print(f"  Target classified_index rows: {target_count} (already processed)")
print(f"  Remaining: {remaining}")
print(f"  FORCE_REPROCESS = {FORCE_REPROCESS}")

Stats loaded from DB (fourth run):
  total_processed (rows):              692
  total_expanded (snippets):           5298
  total_classified_as_clause (snippets): 3298
  total_discussions_extracted (rows):   592
  total_errors (rows):                 0

Fourth run status:
  Source classified_index rows: 2674
  Target classified_index rows: 692 (already processed)
  Remaining: 1982
  FORCE_REPROCESS = False


In [14]:
# Cell 8 — fourth run: Process — Expand, Classify & Extract Discussion
# FIX: commits after EACH row's classified_index INSERT (not per-batch),
# preventing rollbacks from wiping successfully processed rows.
# Uses expand_and_classify and extract_discussion_with_azure
# from second_pipelineoperation (reloaded in previous cell).

def save_stats():
    """Save current stats to stats file."""
    stats['last_updated'] = datetime.now().isoformat()
    with open(STATS_FILE, 'w') as f:
        json.dump(stats, f, indent=2)


def expand_classify_with_retry(conn, doc_id, snippet, max_retries=3):
    """Combined expand+classify with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return expand_and_classify(conn, doc_id, snippet)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


def extract_discussion_with_retry(conn, doc_id, clause, max_retries=3):
    """Extract discussion with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return extract_discussion_with_azure(conn, doc_id, clause)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Discussion retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


# Create staging temp table for per-snippet results (recreated each run)
with target_conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS _staging_snippets")
    cur.execute("""
        CREATE TEMP TABLE _staging_snippets (
            doc_id TEXT, searchquery TEXT,
            snippet_type TEXT, snippet_text TEXT, clause_text TEXT,
            is_contract_clause BOOLEAN, confidence FLOAT, reasoning TEXT
        )
    """)
    target_conn.commit()

print(f"Starting processing (streaming from source, BATCH_SIZE={BATCH_SIZE}, FORCE_REPROCESS={FORCE_REPROCESS})\"")
batch_num = 0

for batch in iter_unprocessed_batches(source_conn, target_conn, BATCH_SIZE, force=FORCE_REPROCESS):
    batch_num += 1

    for row in batch:
        try:
            # Clear staging for this row
            with target_conn.cursor() as cur:
                cur.execute("DELETE FROM _staging_snippets WHERE doc_id = %s AND searchquery = %s",
                            (row['doc_id'], row['searchquery']))
            target_conn.commit()

            # Process matching_columns snippets — iterate inline, write to staging
            for s in insert_data._safe_eval(row['matching_columns_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mc',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Process matching_indents snippets
            for s in insert_data._safe_eval(row['matching_indents_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mi',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Aggregate from staging via SQL — no Python lists
            with target_conn.cursor() as cur:
                cur.execute("""
                    SELECT
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc'),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi'),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        ROUND(AVG(confidence) FILTER (WHERE is_contract_clause)::numeric, 3),
                        (array_agg(reasoning) FILTER (WHERE is_contract_clause))[1],
                        ROUND(AVG(confidence)::numeric, 3),
                        (array_agg(reasoning))[1],
                        COUNT(*) FILTER (WHERE snippet_type='mc'),
                        COUNT(*) FILTER (WHERE snippet_type='mi'),
                        COUNT(*) FILTER (WHERE is_contract_clause),
                        (array_agg(clause_text) FILTER (WHERE is_contract_clause))[1]
                    FROM _staging_snippets
                    WHERE doc_id = %s AND searchquery = %s
                """, (row['doc_id'], row['searchquery']))
                agg = cur.fetchone()

            avg_confidence = float(agg[6] if agg[6] is not None else agg[8] or 0)
            first_reasoning = agg[7] or agg[9] or ''

            # Extract discussion if any classified clauses exist
            extracted_discussion = ''
            sentiment = ''
            sentiment_conf = 0.0
            if agg[13]:  # first_classified_clause exists
                try:
                    d = extract_discussion_with_retry(source_conn, row['doc_id'], agg[13])
                    extracted_discussion = d['discussion']
                    sentiment = d['sentiment']
                    sentiment_conf = d['sentiment_confidence']
                    stats['total_discussions_extracted'] += 1
                except Exception as e:
                    print(f"  Discussion error for doc_id={row['doc_id']}: {e}")
                time.sleep(0.1)

            # Insert into target classified_index
            with target_conn.cursor() as tgt_cur:
                tgt_cur.execute("""
                    INSERT INTO classified_index(
                        Doc_Id, Title, searchquery,
                        matching_indents, matching_columns,
                        matching_columns_after_classification,
                        matching_indents_after_classification,
                        expanded_columns, expanded_indents,
                        expanded_columns_after_classification,
                        expanded_indents_after_classification,
                        extracted_discussion, sentiment,
                        classification_confidence, classification_reasoning,
                        sentiment_confidence
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (Doc_Id, searchquery) DO UPDATE SET
                        Title = EXCLUDED.Title,
                        matching_indents = EXCLUDED.matching_indents,
                        matching_columns = EXCLUDED.matching_columns,
                        matching_columns_after_classification = EXCLUDED.matching_columns_after_classification,
                        matching_indents_after_classification = EXCLUDED.matching_indents_after_classification,
                        expanded_columns = EXCLUDED.expanded_columns,
                        expanded_indents = EXCLUDED.expanded_indents,
                        expanded_columns_after_classification = EXCLUDED.expanded_columns_after_classification,
                        expanded_indents_after_classification = EXCLUDED.expanded_indents_after_classification,
                        extracted_discussion = EXCLUDED.extracted_discussion,
                        sentiment = EXCLUDED.sentiment,
                        classification_confidence = EXCLUDED.classification_confidence,
                        classification_reasoning = EXCLUDED.classification_reasoning,
                        sentiment_confidence = EXCLUDED.sentiment_confidence
                """, (
                    row['doc_id'], row['title'], row['searchquery'],
                    row['matching_indents_raw'], row['matching_columns_raw'],
                    str(agg[2] or []), str(agg[3] or []),
                    str(agg[0] or []), str(agg[1] or []),
                    str(agg[4] or []), str(agg[5] or []),
                    extracted_discussion, sentiment,
                    str(avg_confidence), first_reasoning,
                    str(round(sentiment_conf, 3)),
                ))

            # FIX: commit immediately after each row's INSERT
            # Prevents a later row's error from rolling back this row
            target_conn.commit()

            # Track stats (after commit, so stats only count persisted rows)
            stats['total_expanded'] += (agg[10] or 0) + (agg[11] or 0)
            stats['total_classified_as_clause'] += agg[12] or 0
            stats['total_processed'] += 1

        except Exception as e:
            print(f"  ERROR processing doc_id={row['doc_id']} query={row['searchquery']}: {e}")
            stats['total_errors'] += 1
            target_conn.rollback()  # clear aborted transaction so next row can proceed

    # Save stats after each batch (commit already happened per-row above)
    save_stats()
    print(f"Batch {batch_num} complete "
          f"({stats['total_processed']} total processed, "
          f"{stats['total_discussions_extracted']} discussions, "
          f"{stats['total_errors']} errors)")

print(f"\nProcessing complete. Final stats: {stats}")

Starting processing (streaming from source, BATCH_SIZE=10, FORCE_REPROCESS=False)"
  Retry 1/3 after 1s: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
  Retry 2/3 after 2s: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modif

KeyboardInterrupt: 

In [19]:
# Cell 6/7/7b — fifth run: Load stats from DB + streaming setup
# Combined cell that loads stats directly from the database (not the checkpoint
# file), sets up the streaming generator, and displays remaining work.
# Fifth run: Similar to third and fourth runs, for continued processing of remaining rows.

import importlib
importlib.reload(second_pipelineoperation)
from second_pipelineoperation import expand_and_classify, extract_discussion_with_azure

# Clear any aborted transaction state
source_conn.rollback()
target_conn.rollback()

# ── Load stats from actual DB state ──────────────────────────────────────────

DELIM = "', '"

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    db_processed = cur.fetchone()[0]

    cur.execute("SELECT COUNT(*) FROM classified_index WHERE LENGTH(extracted_discussion) > 0")
    db_discussions = cur.fetchone()[0]

    cur.execute("""SELECT COALESCE(SUM(
        CASE WHEN expanded_columns IS NOT NULL AND LENGTH(expanded_columns) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from expanded_columns), %s), 1)
             ELSE 0 END
        +
        CASE WHEN expanded_indents IS NOT NULL AND LENGTH(expanded_indents) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from expanded_indents), %s), 1)
             ELSE 0 END
    ), 0) FROM classified_index""", (DELIM, DELIM))
    db_expanded = cur.fetchone()[0]

    cur.execute("""SELECT COALESCE(SUM(
        CASE WHEN matching_columns_after_classification IS NOT NULL
                  AND LENGTH(matching_columns_after_classification) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from matching_columns_after_classification), %s), 1)
             ELSE 0 END
        +
        CASE WHEN matching_indents_after_classification IS NOT NULL
                  AND LENGTH(matching_indents_after_classification) > 2
             THEN array_length(string_to_array(
                 trim(both '[]' from matching_indents_after_classification), %s), 1)
             ELSE 0 END
    ), 0) FROM classified_index""", (DELIM, DELIM))
    db_classified = cur.fetchone()[0]

    cur.execute("SELECT COUNT(*) FROM classified_index WHERE classification_reasoning LIKE '%%Error%%'")
    db_errors = cur.fetchone()[0]

stats = {
    'total_processed': db_processed,
    'total_expanded': db_expanded,
    'total_classified_as_clause': db_classified,
    'total_discussions_extracted': db_discussions,
    'total_errors': db_errors,
    'last_updated': datetime.now().isoformat(),
}

with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2)

print("Stats loaded from DB (fifth run):")
print(f"  total_processed (rows):              {db_processed}")
print(f"  total_expanded (snippets):           {db_expanded}")
print(f"  total_classified_as_clause (snippets): {db_classified}")
print(f"  total_discussions_extracted (rows):   {db_discussions}")
print(f"  total_errors (rows):                 {db_errors}")

# ── Streaming generator ──────────────────────────────────────────────────────

FORCE_REPROCESS = False

def _filter_already_processed(target_conn, rows):
    """Filter out rows whose (doc_id, searchquery) already exist in target."""
    if not rows:
        return []
    pairs = [(r[0], r[2]) for r in rows]
    values_placeholder = ", ".join(["(%s, %s)"] * len(pairs))

    query = f"""
        SELECT v.doc_id, v.searchquery
        FROM (VALUES {values_placeholder}) AS v(doc_id, searchquery)
        INNER JOIN classified_index ci
            ON ci.Doc_Id::TEXT = v.doc_id AND ci.searchquery = v.searchquery
    """
    flat_params = [item for pair in pairs for item in pair]
    with target_conn.cursor() as cur:
        cur.execute(query, flat_params)
        existing = set(cur.fetchall())
    return [
        {
            'doc_id': r[0], 'title': r[1], 'searchquery': r[2],
            'matching_indents_raw': r[3], 'matching_columns_raw': r[4],
            'matching_columns_after_classification_raw': r[5],
            'matching_indents_after_classification_raw': r[6],
        }
        for r in rows if (r[0], r[2]) not in existing
    ]


def _all_as_dicts(rows):
    """Return all rows as dicts without filtering (FORCE_REPROCESS mode)."""
    return [
        {
            'doc_id': r[0], 'title': r[1], 'searchquery': r[2],
            'matching_indents_raw': r[3], 'matching_columns_raw': r[4],
            'matching_columns_after_classification_raw': r[5],
            'matching_indents_after_classification_raw': r[6],
        }
        for r in rows
    ]


def iter_unprocessed_batches(source_conn, target_conn, batch_size, force=False):
    """Stream rows from source, yield batches of unprocessed row dicts."""
    with source_conn.cursor(name="stream_classified_index_fifth") as src_cur:
        src_cur.itersize = batch_size
        src_cur.execute("""
            SELECT doc_id::TEXT, title, searchquery, matching_indents, matching_columns,
                   matching_columns_after_classification, matching_indents_after_classification
            FROM classified_index
        """)
        batch = []
        for row in src_cur:
            batch.append(row)
            if len(batch) >= batch_size:
                rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
                batch = []
                if rows:
                    yield rows
        if batch:
            rows = _all_as_dicts(batch) if force else _filter_already_processed(target_conn, batch)
            if rows:
                yield rows


# ── Display remaining work ───────────────────────────────────────────────────

with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_count = cur.fetchone()[0]

with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_count = cur.fetchone()[0]

remaining = source_count - target_count
print(f"\nFifth run status:")
print(f"  Source classified_index rows: {source_count}")
print(f"  Target classified_index rows: {target_count} (already processed)")
print(f"  Remaining: {remaining}")
print(f"  FORCE_REPROCESS = {FORCE_REPROCESS}")

Stats loaded from DB (fifth run):
  total_processed (rows):              870
  total_expanded (snippets):           6889
  total_classified_as_clause (snippets): 4141
  total_discussions_extracted (rows):   723
  total_errors (rows):                 0

Fifth run status:
  Source classified_index rows: 2674
  Target classified_index rows: 870 (already processed)
  Remaining: 1804
  FORCE_REPROCESS = False


In [20]:
# Cell 8 — fifth run: Process — Expand, Classify & Extract Discussion
# FIX: commits after EACH row's classified_index INSERT (not per-batch),
# preventing rollbacks from wiping successfully processed rows.
# Uses expand_and_classify and extract_discussion_with_azure
# from second_pipelineoperation (reloaded in previous cell).

def save_stats():
    """Save current stats to stats file."""
    stats['last_updated'] = datetime.now().isoformat()
    with open(STATS_FILE, 'w') as f:
        json.dump(stats, f, indent=2)


def expand_classify_with_retry(conn, doc_id, snippet, max_retries=3):
    """Combined expand+classify with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return expand_and_classify(conn, doc_id, snippet)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


def extract_discussion_with_retry(conn, doc_id, clause, max_retries=3):
    """Extract discussion with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return extract_discussion_with_azure(conn, doc_id, clause)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Discussion retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


# Create staging temp table for per-snippet results (recreated each run)
with target_conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS _staging_snippets")
    cur.execute("""
        CREATE TEMP TABLE _staging_snippets (
            doc_id TEXT, searchquery TEXT,
            snippet_type TEXT, snippet_text TEXT, clause_text TEXT,
            is_contract_clause BOOLEAN, confidence FLOAT, reasoning TEXT
        )
    """)
    target_conn.commit()

print(f"Starting processing (streaming from source, BATCH_SIZE={BATCH_SIZE}, FORCE_REPROCESS={FORCE_REPROCESS})")
batch_num = 0

for batch in iter_unprocessed_batches(source_conn, target_conn, BATCH_SIZE, force=FORCE_REPROCESS):
    batch_num += 1

    for row in batch:
        try:
            # Clear staging for this row
            with target_conn.cursor() as cur:
                cur.execute("DELETE FROM _staging_snippets WHERE doc_id = %s AND searchquery = %s",
                            (row['doc_id'], row['searchquery']))
            target_conn.commit()

            # Process matching_columns snippets — iterate inline, write to staging
            for s in insert_data._safe_eval(row['matching_columns_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mc',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Process matching_indents snippets
            for s in insert_data._safe_eval(row['matching_indents_raw']):
                result = expand_classify_with_retry(source_conn, row['doc_id'], s)
                with target_conn.cursor() as cur:
                    cur.execute("""INSERT INTO _staging_snippets
                        (doc_id, searchquery, snippet_type, snippet_text, clause_text,
                         is_contract_clause, confidence, reasoning)
                        VALUES (%s,%s,'mi',%s,%s,%s,%s,%s)""",
                        (row['doc_id'], row['searchquery'], s,
                         result['clause_text'], result['is_contract_clause'],
                         result['classification_confidence'], result['classification_reasoning']))
                target_conn.commit()
                del result
                time.sleep(0.1)

            # Aggregate from staging via SQL — no Python lists
            with target_conn.cursor() as cur:
                cur.execute("""
                    SELECT
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc'),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi'),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(snippet_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mc' AND is_contract_clause),
                        array_agg(clause_text) FILTER (WHERE snippet_type='mi' AND is_contract_clause),
                        ROUND(AVG(confidence) FILTER (WHERE is_contract_clause)::numeric, 3),
                        (array_agg(reasoning) FILTER (WHERE is_contract_clause))[1],
                        ROUND(AVG(confidence)::numeric, 3),
                        (array_agg(reasoning))[1],
                        COUNT(*) FILTER (WHERE snippet_type='mc'),
                        COUNT(*) FILTER (WHERE snippet_type='mi'),
                        COUNT(*) FILTER (WHERE is_contract_clause),
                        (array_agg(clause_text) FILTER (WHERE is_contract_clause))[1]
                    FROM _staging_snippets
                    WHERE doc_id = %s AND searchquery = %s
                """, (row['doc_id'], row['searchquery']))
                agg = cur.fetchone()

            avg_confidence = float(agg[6] if agg[6] is not None else agg[8] or 0)
            first_reasoning = agg[7] or agg[9] or ''

            # Extract discussion if any classified clauses exist
            extracted_discussion = ''
            sentiment = ''
            sentiment_conf = 0.0
            if agg[13]:  # first_classified_clause exists
                try:
                    d = extract_discussion_with_retry(source_conn, row['doc_id'], agg[13])
                    extracted_discussion = d['discussion']
                    sentiment = d['sentiment']
                    sentiment_conf = d['sentiment_confidence']
                    stats['total_discussions_extracted'] += 1
                except Exception as e:
                    print(f"  Discussion error for doc_id={row['doc_id']}: {e}")
                time.sleep(0.1)

            # Insert into target classified_index
            with target_conn.cursor() as tgt_cur:
                tgt_cur.execute("""
                    INSERT INTO classified_index(
                        Doc_Id, Title, searchquery,
                        matching_indents, matching_columns,
                        matching_columns_after_classification,
                        matching_indents_after_classification,
                        expanded_columns, expanded_indents,
                        expanded_columns_after_classification,
                        expanded_indents_after_classification,
                        extracted_discussion, sentiment,
                        classification_confidence, classification_reasoning,
                        sentiment_confidence
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (Doc_Id, searchquery) DO UPDATE SET
                        Title = EXCLUDED.Title,
                        matching_indents = EXCLUDED.matching_indents,
                        matching_columns = EXCLUDED.matching_columns,
                        matching_columns_after_classification = EXCLUDED.matching_columns_after_classification,
                        matching_indents_after_classification = EXCLUDED.matching_indents_after_classification,
                        expanded_columns = EXCLUDED.expanded_columns,
                        expanded_indents = EXCLUDED.expanded_indents,
                        expanded_columns_after_classification = EXCLUDED.expanded_columns_after_classification,
                        expanded_indents_after_classification = EXCLUDED.expanded_indents_after_classification,
                        extracted_discussion = EXCLUDED.extracted_discussion,
                        sentiment = EXCLUDED.sentiment,
                        classification_confidence = EXCLUDED.classification_confidence,
                        classification_reasoning = EXCLUDED.classification_reasoning,
                        sentiment_confidence = EXCLUDED.sentiment_confidence
                """, (
                    row['doc_id'], row['title'], row['searchquery'],
                    row['matching_indents_raw'], row['matching_columns_raw'],
                    str(agg[2] or []), str(agg[3] or []),
                    str(agg[0] or []), str(agg[1] or []),
                    str(agg[4] or []), str(agg[5] or []),
                    extracted_discussion, sentiment,
                    str(avg_confidence), first_reasoning,
                    str(round(sentiment_conf, 3)),
                ))

            # FIX: commit immediately after each row's INSERT
            # Prevents a later row's error from rolling back this row
            target_conn.commit()

            # Track stats (after commit, so stats only count persisted rows)
            stats['total_expanded'] += (agg[10] or 0) + (agg[11] or 0)
            stats['total_classified_as_clause'] += agg[12] or 0
            stats['total_processed'] += 1

        except Exception as e:
            print(f"  ERROR processing doc_id={row['doc_id']} query={row['searchquery']}: {e}")
            stats['total_errors'] += 1
            target_conn.rollback()  # clear aborted transaction so next row can proceed

    # Save stats after each batch (commit already happened per-row above)
    save_stats()
    print(f"Batch {batch_num} complete "
          f"({stats['total_processed']} total processed, "
          f"{stats['total_discussions_extracted']} discussions, "
          f"{stats['total_errors']} errors)")

print(f"\nProcessing complete. Final stats: {stats}")

Starting processing (streaming from source, BATCH_SIZE=10, FORCE_REPROCESS=False)
  Retry 1/3 after 1s: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
  Retry 2/3 after 2s: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify

In [21]:
# Cell 9: Verification

# Row counts in target
with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_ci_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results")
    target_sr_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM search_queries")
    target_sq_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM classified_index WHERE LENGTH(extracted_discussion) > 0")
    target_disc_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM classified_index WHERE classification_confidence IS NOT NULL AND classification_confidence != ''")
    target_conf_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results WHERE court_name IS NOT NULL AND court_name != ''")
    target_meta_count = cur.fetchone()[0]

# Row counts in source
with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_ci_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results")
    source_sr_count = cur.fetchone()[0]

print("=== Row Count Comparison ===")
print(f"  classified_index:  source={source_ci_count}  target={target_ci_count}")
print(f"  stored_results:    source={source_sr_count}  target={target_sr_count}")
print(f"  search_queries:    target={target_sq_count}")
print(f"  with discussion:   target={target_disc_count}")
print(f"  with confidence:   target={target_conf_count}")
print(f"  with court meta:   target={target_meta_count}")

# Sentiment distribution with confidence
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT sentiment, COUNT(*),
               AVG(CASE WHEN sentiment_confidence != '' THEN CAST(sentiment_confidence AS FLOAT) ELSE NULL END)
        FROM classified_index
        WHERE LENGTH(extracted_discussion) > 0
        GROUP BY sentiment
    """)
    sentiment_dist = cur.fetchall()

if sentiment_dist:
    print("\n=== Sentiment Distribution (with avg confidence) ===")
    for sentiment, count, avg_conf in sentiment_dist:
        avg_str = f"{avg_conf:.2f}" if avg_conf else "N/A"
        print(f"  {sentiment or '(empty)'}: {count} (avg confidence: {avg_str})")

# Classification confidence distribution
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT 
            CASE 
                WHEN CAST(classification_confidence AS FLOAT) >= 0.8 THEN 'high (>=0.8)'
                WHEN CAST(classification_confidence AS FLOAT) >= 0.5 THEN 'medium (0.5-0.8)'
                ELSE 'low (<0.5)'
            END as conf_bucket,
            COUNT(*)
        FROM classified_index
        WHERE classification_confidence IS NOT NULL AND classification_confidence != ''
        GROUP BY conf_bucket
    """)
    conf_dist = cur.fetchall()

if conf_dist:
    print("\n=== Classification Confidence Distribution ===")
    for bucket, count in conf_dist:
        print(f"  {bucket}: {count}")

# Show sample rows with all new fields
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT ci.doc_id, ci.searchquery, ci.sentiment,
               ci.classification_confidence, ci.classification_reasoning,
               ci.sentiment_confidence,
               sr.court_name, sr.judgment_date, sr.case_citation
        FROM classified_index ci
        LEFT JOIN stored_results sr ON ci.doc_id = sr.Doc_ID
        WHERE LENGTH(ci.extracted_discussion) > 0
        LIMIT 10
    """)
    verification_rows = cur.fetchall()

if verification_rows:
    print("\n=== Sample Rows with All New Fields ===")
    for r in verification_rows:
        print(f"  DocID: {r[0]}, Query: {r[1]}")
        print(f"    Sentiment: {r[2]}, Confidence: {r[5]}")
        print(f"    Classification Confidence: {r[3]}, Reasoning: {(r[4] or '')[:80]}")
        print(f"    Court: {r[6]}, Date: {r[7]}, Citation: {(r[8] or '')[:60]}")
        print()
else:
    print("\nNo rows with extracted discussions found.")

# Close connections
source_conn.close()
target_conn.close()
print("\nBoth database connections closed.")

=== Row Count Comparison ===
  classified_index:  source=2674  target=917
  stored_results:    source=835  target=835
  search_queries:    target=822
  with discussion:   target=768
  with confidence:   target=917
  with court meta:   target=667

=== Sentiment Distribution (with avg confidence) ===
  neutral or positive: 535 (avg confidence: 0.82)
  (empty): 11 (avg confidence: N/A)
  negative or struck down: 222 (avg confidence: 0.83)

=== Classification Confidence Distribution ===
  low (<0.5): 47
  high (>=0.8): 781
  medium (0.5-0.8): 89

=== Sample Rows with All New Fields ===
  DocID: 46122284, Query: ip licensing
    Sentiment: neutral or positive, Confidence: 0.8
    Classification Confidence: 0.943, Reasoning: This clause assigns property development rights from the owner to THDC, includin
    Court: Income Tax Appellate Tribunal - Chandigarh, Date: 2011-10-20, Citation: IT-188-CHANDI-2013

  DocID: 1282775, Query: artist licensing
    Sentiment: neutral or positive, Confidenc